# Ablação Ridge multialvo: TU × TUO × TUOV

Este notebook acrescenta a velocidade do vento ao conjunto TUO e verifica se o conjunto **TUOV** melhora a previsão direta de temperatura e umidade de `t+1` a `t+24`.

## Regra experimental

- TU e TUO são reestimados no mesmo ambiente para assegurar comparação homogênea.
- O melhor `alpha` de cada configuração é escolhido somente em 2024.
- A configuração final é escolhida somente pela validação de 2024.
- 2025 permanece teste fechado; 2026 parcial é avaliação adicional.
- Não são usadas covariáveis meteorológicas futuras reais.
- A comparação considera ganho preditivo e custo de complexidade.

## 1. Importações e configuração

In [ ]:
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

DIRETORIO_DADOS = Path("dados_processados_multivariados")
DIRETORIO_SAIDA = Path("resultados_modelagem/ablacao_ridge_TU_TUO_TUOV")
DIRETORIO_SAIDA.mkdir(parents=True, exist_ok=True)

CONFIGURACOES = ["TU", "TUO", "TUOV"]
SPLITS = ["treino", "validacao", "teste", "avaliacao_2026"]
ALPHAS = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]
ALVOS = ["temperatura_c", "umidade_pct"]
UNIDADES = {"temperatura_c": "°C", "umidade_pct": "p.p."}
TOLERANCIA_PIORA_ALVO_PCT = 1.0
GANHO_MINIMO_COMPLEXIDADE_PCT = 0.25

print("Este experimento usa CPU com scikit-learn.")
print("alpha não possui unidade; MAE/RMSE finais usam °C e pontos percentuais.")

## 2. Carregamento dos três conjuntos

In [ ]:
def carregar_configuracao(config):
    pasta = DIRETORIO_DADOS / config
    nomes = ["treino", "validacao", "teste", "avaliacao_2026"]
    obrigatorios = [pasta/f"{s}.npz" for s in nomes] + [pasta/"scaler_y.joblib", pasta/"metadados.json"]
    faltantes = [str(x) for x in obrigatorios if not x.exists()]
    if faltantes:
        raise FileNotFoundError("Arquivos ausentes:" + chr(10) + chr(10).join(faltantes))
    splits={}
    for s in nomes:
        with np.load(pasta/f"{s}.npz") as arq:
            splits[s]={k:arq[k] for k in arq.files}
    with (pasta/"metadados.json").open(encoding="utf-8") as f:
        meta=json.load(f)
    assert meta["usa_covariaveis_meteorologicas_futuras"] is False
    return {"splits":splits,"scaler_y":joblib.load(pasta/"scaler_y.joblib"),"meta":meta}

artefatos={c:carregar_configuracao(c) for c in CONFIGURACOES}
for c in CONFIGURACOES:
    print(c, artefatos[c]["meta"]["colunas_meteorologicas_historicas"], artefatos[c]["meta"]["contagens"])

## 3. Preparação tabular

As 24 horas históricas são achatadas sem perder a ordem e concatenadas aos atributos temporais futuros. TU, TUO e TUOV resultam em números diferentes de colunas, mas mantêm os mesmos 48 alvos diretos.

In [ ]:
def preparar_X(split):
    hist=split["X_historico"].reshape(len(split["X_historico"]),-1)
    fut=split["X_futuro_temporal"].reshape(len(split["X_futuro_temporal"]),-1)
    return np.concatenate([hist,fut],axis=1).astype(np.float32,copy=False)

def preparar_y(split):
    return split["y"].reshape(len(split["y"]),-1).astype(np.float32,copy=False)

matrizes={}
for c in CONFIGURACOES:
    matrizes[c]={
        "X":{s:preparar_X(artefatos[c]["splits"][s]) for s in SPLITS},
        "y":{s:preparar_y(artefatos[c]["splits"][s]) for s in SPLITS},
    }
    print(c,"X treino",matrizes[c]["X"]["treino"].shape,"y treino",matrizes[c]["y"]["treino"].shape)

## 4. Funções de avaliação

In [ ]:
def remodelar(y): return y.reshape(-1,24,2)

def inverter(y_norm,scaler):
    forma=y_norm.shape
    return scaler.inverse_transform(y_norm.reshape(-1,2)).reshape(forma)

def resumo(y_real,y_pred,config,split,alpha):
    linhas=[]
    for j,alvo in enumerate(ALVOS):
        real=y_real[:,:,j].ravel(); pred=y_pred[:,:,j].ravel()
        linhas.append({"configuracao":config,"split":split,"alpha":alpha,"alvo":alvo,"unidade":UNIDADES[alvo],
                       "mae":mean_absolute_error(real,pred),"rmse":mean_squared_error(real,pred)**0.5,
                       "vies":float(np.mean(pred-real)),"n":len(real)})
    return pd.DataFrame(linhas)

def por_horizonte(y_real,y_pred,config,split,alpha):
    linhas=[]
    for h in range(24):
        for j,alvo in enumerate(ALVOS):
            real=y_real[:,h,j]; pred=y_pred[:,h,j]
            linhas.append({"configuracao":config,"split":split,"alpha":alpha,"horizonte_horas":h+1,
                           "alvo":alvo,"unidade":UNIDADES[alvo],"mae":mean_absolute_error(real,pred),
                           "rmse":mean_squared_error(real,pred)**0.5,"vies":float(np.mean(pred-real)),"n":len(real)})
    return pd.DataFrame(linhas)

def criterio(y_real,y_pred,scaler):
    mt=mean_absolute_error(y_real[:,:,0].ravel(),y_pred[:,:,0].ravel())
    mu=mean_absolute_error(y_real[:,:,1].ravel(),y_pred[:,:,1].ravel())
    pt=mt/scaler.scale_[0]; pu=mu/scaler.scale_[1]
    return (pt+pu)/2,pt,pu,mt,mu

## 5. Treinamento visível e seleção do `alpha` em 2024

In [ ]:
linhas=[]; modelos={}
for c in CONFIGURACOES:
    print(chr(10)+"="*72); print("CONFIGURAÇÃO",c); print("="*72)
    X=matrizes[c]["X"]; y=matrizes[c]["y"]; scaler=artefatos[c]["scaler_y"]
    yt=inverter(remodelar(y["treino"]),scaler); yv=inverter(remodelar(y["validacao"]),scaler)
    for i,a in enumerate(ALPHAS,1):
        print(f"[{i}/{len(ALPHAS)}] Treinando {c}, alpha={a}...")
        t0=time.perf_counter(); m=Ridge(alpha=a,fit_intercept=True,solver="lsqr",tol=1e-4); m.fit(X["treino"],y["treino"]); dt=time.perf_counter()-t0
        pt=inverter(remodelar(m.predict(X["treino"])),scaler); pv=inverter(remodelar(m.predict(X["validacao"])),scaler)
        cv,padt,padu,mvt,mvu=criterio(yv,pv,scaler); _,_,_,mtt,mtu=criterio(yt,pt,scaler)
        linhas.append({"configuracao":c,"alpha":a,"tempo_segundos":dt,"mae_treino_temperatura_c":mtt,
                       "mae_validacao_temperatura_c":mvt,"mae_treino_umidade_pct":mtu,"mae_validacao_umidade_pct":mvu,
                       "mae_padronizado_temperatura":padt,"mae_padronizado_umidade":padu,"criterio_padronizado_medio":cv})
        modelos[(c,a)]=m
        print(f"  {dt:.1f}s | treino {mtt:.3f} °C, {mtu:.3f} p.p. | validação {mvt:.3f} °C, {mvu:.3f} p.p. | critério {cv:.6f}")

selecao=pd.DataFrame(linhas)
melhores=(selecao.sort_values(["configuracao","criterio_padronizado_medio","alpha"]).groupby("configuracao",as_index=False).first())
display(melhores)

## 6. Avaliação dos melhores modelos

In [ ]:
globais=[]; horizontes=[]
for _,r in melhores.iterrows():
    c=r.configuracao; a=float(r.alpha); m=modelos[(c,a)]; scaler=artefatos[c]["scaler_y"]
    for s in SPLITS:
        yr=inverter(remodelar(matrizes[c]["y"][s]),scaler)
        yp=inverter(remodelar(m.predict(matrizes[c]["X"][s])),scaler)
        globais.append(resumo(yr,yp,c,s,a)); horizontes.append(por_horizonte(yr,yp,c,s,a))
metricas_globais=pd.concat(globais,ignore_index=True)
metricas_horizonte=pd.concat(horizontes,ignore_index=True)
display(metricas_globais.sort_values(["split","alvo","configuracao"]).reset_index(drop=True))

## 7. Ganho incremental de TUO e TUOV

In [ ]:
def comparar(base,candidata):
    b=metricas_globais[metricas_globais.configuracao==base].set_index(["split","alvo"])
    c=metricas_globais[metricas_globais.configuracao==candidata].set_index(["split","alvo"])
    out=[]
    for idx in b.index:
        split,alvo=idx
        out.append({"base":base,"candidata":candidata,"split":split,"alvo":alvo,"unidade":b.loc[idx,"unidade"],
                    "mae_base":b.loc[idx,"mae"],"mae_candidata":c.loc[idx,"mae"],
                    "ganho_mae_pct":100*(b.loc[idx,"mae"]-c.loc[idx,"mae"])/b.loc[idx,"mae"],
                    "rmse_base":b.loc[idx,"rmse"],"rmse_candidata":c.loc[idx,"rmse"],
                    "ganho_rmse_pct":100*(b.loc[idx,"rmse"]-c.loc[idx,"rmse"])/b.loc[idx,"rmse"]})
    return pd.DataFrame(out)

comparacoes=pd.concat([comparar("TU","TUO"),comparar("TUO","TUOV")],ignore_index=True)
display(comparacoes.sort_values(["base","split","alvo"]).reset_index(drop=True))

## 8. Gráficos de validação por horizonte

In [ ]:
for alvo in ALVOS:
    fig,ax=plt.subplots(figsize=(8,4.5))
    for c in CONFIGURACOES:
        d=metricas_horizonte[(metricas_horizonte.split=="validacao")&(metricas_horizonte.alvo==alvo)&(metricas_horizonte.configuracao==c)]
        ax.plot(d.horizonte_horas,d.mae,marker="o",markersize=3,label=c)
    ax.set(title=f"MAE por horizonte — validação 2024 — {alvo}",xlabel="Horizonte (horas)",ylabel=f"MAE ({UNIDADES[alvo]})")
    ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.show()

## 9. Regra de decisão com penalidade de complexidade

TUOV somente será adotado se:

1. reduzir o critério padronizado médio de TUO em pelo menos 0,25%;
2. não piorar o MAE de nenhum alvo em mais de 1%;
3. a decisão for tomada exclusivamente em 2024.

O limiar mínimo evita acrescentar vento por um ganho numericamente desprezível.

In [ ]:
crit={r.configuracao:r.criterio_padronizado_medio for _,r in melhores.iterrows()}
ganho_criterio_tuov=100*(crit["TUO"]-crit["TUOV"])/crit["TUO"]
val=comparacoes[(comparacoes.base=="TUO")&(comparacoes.candidata=="TUOV")&(comparacoes.split=="validacao")]
sem_piora=(val.ganho_mae_pct>=-TOLERANCIA_PIORA_ALVO_PCT).all()
RECOMENDACAO="TUOV" if ganho_criterio_tuov>=GANHO_MINIMO_COMPLEXIDADE_PCT and sem_piora else "TUO"
print(f"Ganho do critério TUOV sobre TUO: {ganho_criterio_tuov:.4f}%")
print("Sem piora superior à tolerância por alvo:",sem_piora)
print("CONFIGURAÇÃO RECOMENDADA:",RECOMENDACAO)

## 10. Salvamento e verificação final

In [ ]:
selecao.to_csv(DIRETORIO_SAIDA/"selecao_alpha_TU_TUO_TUOV.csv",index=False,encoding="utf-8-sig")
melhores.to_csv(DIRETORIO_SAIDA/"melhores_TU_TUO_TUOV.csv",index=False,encoding="utf-8-sig")
metricas_globais.to_csv(DIRETORIO_SAIDA/"metricas_globais_TU_TUO_TUOV.csv",index=False,encoding="utf-8-sig")
metricas_horizonte.to_csv(DIRETORIO_SAIDA/"metricas_horizonte_TU_TUO_TUOV.csv",index=False,encoding="utf-8-sig")
comparacoes.to_csv(DIRETORIO_SAIDA/"comparacoes_incrementais.csv",index=False,encoding="utf-8-sig")
for _,r in melhores.iterrows():
    joblib.dump(modelos[(r.configuracao,float(r.alpha))],DIRETORIO_SAIDA/f"modelo_ridge_{r.configuracao}.joblib")
meta={"recomendacao_validacao_2024":RECOMENDACAO,"ganho_criterio_TUOV_sobre_TUO_pct":ganho_criterio_tuov,
      "ganho_minimo_complexidade_pct":GANHO_MINIMO_COMPLEXIDADE_PCT,"tolerancia_piora_alvo_pct":TOLERANCIA_PIORA_ALVO_PCT,
      "usa_covariaveis_meteorologicas_futuras":False,"processamento":"CPU - scikit-learn Ridge",
      "versoes":{"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,"plataforma":platform.platform()}}
with (DIRETORIO_SAIDA/"metadados_ablacao.json").open("w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
assert np.isfinite(metricas_globais[["mae","rmse","vies"]].to_numpy()).all()
assert meta["usa_covariaveis_meteorologicas_futuras"] is False
print("VERIFICAÇÃO FINAL: OK")
print("Recomendação pela validação de 2024:",RECOMENDACAO)

## Encerramento

Após revisar a ablação, a configuração escolhida será usada no próximo modelo não linear. Se TUOV não superar o ganho mínimo, TUO será mantido por parcimônia.